metageについてモデル化  
手順は144にある通り

In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from IPython.display import display
import time

import sys
from decimal import Decimal, ROUND_HALF_UP
import re
#For exporting .pdf file with editable text
import matplotlib
matplotlib.rcParams['pdf.fonttype']=42
matplotlib.rcParams['ps.fonttype']=42

import collections

#!pip install MissForest
from missforest import MissForest
from sklearn.ensemble import RandomForestRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#!conda install -c conda-forge -y watermark
%load_ext watermark
%watermark -a 'Akari FUJIMOTO' -udnt -vmw -iv

In [ ]:
japanize_matplotlib.japanize()

print(plt.rcParams['font.family'])

# 1. data preparation

## 1-1. import transform skew met date

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'metDF_2k-raw.tsv'

tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF #本来ならmet

In [ ]:
#Save cohort metadata
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_2k.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF

In [ ]:
#metの人数を揃える
tempDF1 = metDF.copy()
tempDF2 = cohortDF

#Clean
tempDF1 = tempDF1.loc[tempDF1.index.isin(tempDF2.index)]

display(tempDF1)
display(tempDF1.describe(include='all'))

metDF = tempDF1#Update

# 1. transform to less skewness

In [ ]:
tempDF = metDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
#Check negative or zero value prior to transformation
print('Minimum:', metDF.min(axis=None))

In [ ]:
tempDF = metDF.copy()
skew_cutoff = 1.5

#Log-transformation if skewness is over cutoff
log_cols = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    skewness = stats.skew(tempS)
    if skewness > skew_cutoff:
        log_cols.append(col_n)
        tempDF[col_n] = np.log(tempDF[col_n]+1)

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF#Update

In [ ]:
tempDF = metDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks',  context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

# 2. random forest imputation

In [ ]:
tempDF = metDF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

In [ ]:
tempDF = metDF.copy()

#RF imputation
imputer = MissForest(categorical=None,
                     max_iter=5,
                     rgr=RandomForestRegressor(n_estimators=150,
                                               criterion='squared_error',
                                               max_features='sqrt',
                                               max_depth=None,
                                               n_jobs=-1,
                                               random_state=123))
t_start = time.time()
tempA = imputer.fit_transform(tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)

metDF_RF = tempDF

In [ ]:
tempDF = metDF_RF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

imputerでdiscoveryのパラメータを保持しておく

# save RF data

In [ ]:
#Save data without NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '261002_TMM-BAge_MetAge_'
fileName = 'metDF-RFimputed.tsv'
tempDF = metDF_RF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# 3. validation prepare

skew済みで条件は同じ。

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'metDF_1k-raw.tsv'

tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

metDF = tempDF

In [ ]:
#Save data with NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '260924_TMM-BAge_PreprocessMetDF-CommCohort67K-ver1_'
fileName = 'demographicsDF_1k.tsv'
tempDF = pd.read_csv(fileDir+ipynbName+fileName, sep='\t')
tempDF = tempDF.set_index('ParticipantID')

display(tempDF)
display(tempDF.describe(include='all'))

cohortDF = tempDF #本来ならmet

In [ ]:
tempDF = metDF

#Check skewness
tempL = []
for col_n in tempDF.columns:
    tempS = tempDF[col_n].dropna()
    tempL.append(stats.skew(tempS))
tempS = pd.Series(tempL, index=tempDF.columns, name='Skewness')
tempS = tempS.sort_values(ascending=True)

display(tempS)
display(tempS.describe())

sns.set(style='ticks', context='talk')
plt.figure(figsize=(4, 3))
sns.distplot(tempS)
sns.despine()
plt.ylabel('Density')
plt.xlabel('Skewness')
plt.show()

In [ ]:
tempDF = metDF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

In [ ]:
tempDF = metDF.copy()

t_start = time.time()
tempA = imputer.transform(tempDF)
t_elapsed = time.time() - t_start
print('Elapsed time:', round(t_elapsed//60), 'min', round(t_elapsed%60, 1), 'sec')

#Clean
tempDF = pd.DataFrame(data=tempA, index=tempDF.index, columns=tempDF.columns)

metDF_RF = tempDF

In [ ]:
tempDF = metDF_RF

#Check
count = tempDF.isnull().to_numpy().sum(axis=None)
nrows, ncols = tempDF.shape
print('nrows:', nrows)
print('ncols:', ncols)
print('nrows*ncols:', nrows*ncols)
print('nNaNs:', count, '(', count/(nrows*ncols)*100, '[%])')
print('Mean nNaNs per row:', count/nrows)
print('Mean nNaNs per column:', count/ncols)

# save

In [ ]:
#Save data without NAs
fileDir = './ExportData/Preprocess/metabolome/'
ipynbName = '261002_TMM-BAge_MetAge_'
fileName = 'validation-metDF-RFimputed.tsv'
tempDF = metDF_RF
tempDF.to_csv(fileDir+ipynbName+fileName, sep='\t', index=True)

display(tempDF)
display(tempDF.describe(include='all'))

# end of notebook